In [1]:
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import re
from pypinyin import lazy_pinyin
from rapidfuzz import fuzz
import math
from uuid import uuid4 as uuid
from dotenv import load_dotenv
import subprocess
from tqdm import tqdm
import json
load_dotenv("../.env")

root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
df = pd.read_csv("songs.csv")
df.tail(5)

,code,type,title,lyrics,pinyin
498,UNK-13,Hymn,恩爱救赎主,各各他山上，悲苦之晨，主憔悴独行，疲倦苦辛；\n为救赎罪人，十架牺牲，免人永迷失，大施恩拯。...,ge ge ta shan shang bei ku zhi chen zhu qiao c...
499,UNK-14,Hymn,可叹我主圣体血流,可叹我主，圣体血流，\n为我受死衷痛，\n情愿舍命，赎我愆尤，\n我仍卑微无用。\n\n在我...,ke tan wo zhu sheng ti xue liu wei wo shou si ...
500,ZMS-177,Hymn,邦国帝王兴亡代谢,邦国帝王兴亡代谢\n回首如今安在\n唯主教会千年一日\n祷声依旧不衰\n\n从外看她根基稳固...,bang guo di wang xing wang dai xie hui shou ru...
501,UNK-15,PnW,荣美的救主,耶稣，荣美的救主，复活全能真神，\n荣耀君王，神羔羊，圣洁和公义。\n\n尊荣的拯救者，明亮...,ye su rong mei de jiu zhu fu huo quan neng zhe...
502,UNK-16,PnW,十字架,耶稣背负十架 为我担当过犯毫无保留为我 舍命在十架上祢受鞭伤我得医治 祢受刑罚我得自由释放十...,ye su bei fu shi jia wei wo dan dang guo fan h...


In [3]:
files = sorted([f"{d}/{f}" for d in os.listdir(root) for f in os.listdir(f"{root}/{d}")])

to_rename = {}
for i, row in df.iterrows():
    title = row.title
    title_ = re.sub(r"[，。！？、“”：；\n]", "", row.title).replace("赞美诗24", "")
    if title_ == row.title:
        continue
    df.at[i, "title"] = title_
    if not (affected := [f for f in files if title in f]):
        continue
    print(f">>>>>>>>>> {row.name}: {row.title}")
    print("\n".join(affected))
    for f in affected:
        new_name = to_rename.get(f, f).replace(title, title_)
        to_rename[f] = new_name

In [ ]:
folders = set()
for old_path, new_path in to_rename.items():
    d, _ = old_path.split("/", 1)
    folders.add(d)
    os.rename(f"{root}/{old_path}", f"{root}/{new_path}")

# for d in folders:
#     !sudo -u www-data php /var/www/html/nextcloud_sacm/occ files:scan --path sacm.av/files/Recordings/{d}

In [3]:
def pinyin(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    result = " ".join(lazy_pinyin(text))
    return re.sub(r"\s+", " ", result).strip()


def windows(tokens, size, step):
    if len(tokens) <= size:
        yield " ".join(tokens)
    else:
        for i in range(0, len(tokens) - size + 1, step):
            yield " ".join(tokens[i:i+size])


def best_window_score(query_py, lyrics_py, size=50, step=10):
    query_tokens = query_py.split()
    lyric_tokens = lyrics_py.split()
    score = max(
        fuzz.ratio(qw, lw)
        for qw in windows(query_tokens, size, step)
        for lw in windows(lyric_tokens, size, step)
    )
    return score / 100


def get_duration(filepath) -> float:
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            filepath,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"[ERROR] ffprobe failed: {result.stderr}")
        return 0
    return float(result.stdout.strip())


def split_to_limit(filepath, limit=26_214_400, margin=0.90, out_dir="tmp"):
    size = os.path.getsize(filepath)
    if size <= limit:
        return [filepath]
    os.makedirs(out_dir, exist_ok=True)
    duration = get_duration(filepath)
    bitrate_kbps = 128
    chunk_seconds = max(1, int(limit * margin * 8 / (bitrate_kbps * 1000)))
    chunk_paths = []
    
    for start in range(0, math.ceil(duration), chunk_seconds):
        chunk_path = os.path.join(out_dir, f"{uuid()}.mp3")
        subprocess.run([
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-t", str(chunk_seconds),
            "-i", filepath,
            "-vn",
            "-c:a", "libmp3lame",
            "-b:a", f"{bitrate_kbps}k",
            chunk_path,
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        chunk_paths.append(chunk_path)
        print(
            f"chunk {len(chunk_paths)}: "
            f"{os.path.getsize(chunk_path):,} bytes "
            f"(limit {limit:,}) -> {chunk_path}"
        )
    return chunk_paths

In [3]:
def match_zoom_to_sq(d, tol=10, verbose=False):
    files = sorted(os.listdir(f"{root}/{d}"))
    zoom_files = [f for f in files if f.startswith("ZOOM")]
    sq_files = [f for f in files if not f.startswith("ZOOM")]
    
    if not zoom_files or not sq_files:
        return {}

    durations = {f: int(get_duration(f"{root}/{d}/{f}")) for f in files}
    if verbose:
        print(json.dumps(durations, indent=2, ensure_ascii=False))

    reduced_sq_files = []
    for f in sq_files:
        other_files = [x for x in reduced_sq_files if not x.startswith("SQ")]
        if f.startswith("SQ") or not any(durations[f] == durations[x] for x in other_files):
            reduced_sq_files.append(f)
    sq_files = reduced_sq_files

    if verbose:
        print(zoom_files)
        print(sq_files)

    # If same number of files, just pair in order
    if len(zoom_files) == len(sq_files):
        ordered_matches = dict(zip(zoom_files, sq_files))
        # Make sure the sizes match though
        if all(abs(durations[z] - durations[s]) < tol for z, s in ordered_matches.items()):
            return ordered_matches

    # Otherwise, take the largest filesize (P&W) pair as reference
    matches = {}

    def traverse(ref_zoom_idx, ref_sq_idx, zoom_list, sq_list):
        z_idx = ref_zoom_idx + 1
        s_idx = ref_sq_idx + 1
        while z_idx < len(zoom_list) and s_idx < len(sq_list):
            zoom_file = zoom_list[z_idx]
            for i, sq_file in enumerate(sq_list):
                if sq_file.startswith("SQ") and i < s_idx:
                    continue
                if verbose:
                    print(f"{i=} {s_idx=} {zoom_file} ({durations[zoom_file]}) : {sq_file} ({durations[sq_file]})")
                if abs(durations[zoom_file] - durations[sq_file]) > tol:
                    continue
                matches[zoom_file] = sq_file
                if sq_file.startswith("SQ"):
                    s_idx = i + 1
                break
            z_idx += 1

    def prune_same_values(d):
        counts = {v: len([k for k in d if d[k] == v]) for v in d.values()}
        return {k: v for k, v in d.items() if counts[v] == 1}

    largest_zoom_file = max(zoom_files, key=lambda f: durations[f])
    largest_sq_file = max(sq_files, key=lambda f: durations[f])
    if not largest_sq_file.startswith("SQ"):
        # since order can't be gleaned from filename, just traverse the whole list
        traverse(-1, -1, zoom_files, sq_files)
        return prune_same_values(matches)

    if abs(durations[largest_zoom_file] - durations[largest_sq_file]) <= tol:
        matches[largest_zoom_file] = largest_sq_file
        zoom_idx = zoom_files.index(largest_zoom_file)
        sq_idx = sq_files.index(largest_sq_file)
    else:
        cost = np.array([
            [abs(durations[z] - durations[s]) for s in sq_files]
            for z in zoom_files
        ])
        zoom_idx, sq_idx = np.unravel_index(np.argmin(cost), cost.shape)
        matches[zoom_files[zoom_idx]] = sq_files[sq_idx]

    traverse(zoom_idx, sq_idx, zoom_files, sq_files)
    traverse(len(zoom_files) - zoom_idx - 1, len(sq_files) - sq_idx - 1, zoom_files[::-1], sq_files[::-1])

    return prune_same_values(matches)

# matches = match_zoom_to_sq("2026-04-11")
# print(json.dumps(matches, indent=2, ensure_ascii=False))

In [ ]:
lyrics = """耶稣背负十架 为我担当过犯毫无保留为我 舍命在十架上祢受鞭伤我得医治 祢受刑罚我得自由释放十字架 十字架 耶稣以爱覆盖我十字架 十字架 祢宝血为我流下十字架 十字架 我得救赎的记号十字架 十字架 是我永远的荣耀十字架 十字架 永是我的荣耀我众罪都洗清洁 唯靠耶稣宝血"""
num = max([int(code.split("-")[1]) for code in df.loc[df.code.str.startswith("UNK")].code])
df.loc[len(df)] = {
    "code": f"UNK-{num + 1}",
    "type": "PnW",
    "title": "十字架",
    "lyrics": lyrics,
    "pinyin": pinyin(lyrics),
}
# idx = 495
# df.loc[idx, "lyrics"] = lyrics
# df.loc[idx, "pinyin"] = pinyin(lyrics)
df.to_csv("songs.csv", index=False)
df.tail(1)

,code,type,title,lyrics,pinyin
502,UNK-16,PnW,十字架,耶稣背负十架 为我担当过犯毫无保留为我 舍命在十架上祢受鞭伤我得医治 祢受刑罚我得自由释放十...,ye su bei fu shi jia wei wo dan dang guo fan h...


In [46]:
for d in sorted(os.listdir(root), reverse=True):
    if not d.startswith("2023-0"):
        continue
    files = sorted(os.listdir(f"{root}/{d}"))
    for f in files:
        if bool(re.search(r'[\u4e00-\u9fff]', f)):
            continue
        filepath = f"{root}/{d}/{f}"
        # titles = [substr for substr in f.sp
        # lit(".")[0].split("_") if bool(re.search(r'[\u4e00-\u9fff]', substr))]
        duration = get_duration(filepath)
        if duration < 30:
            continue
        # approx_num_songs = 2
        # if len(titles) < approx_num_songs:
        mins, secs = int(duration // 60), int(duration % 60)
        print(f"{filepath} - {mins:02d}:{secs:02d}")

/mnt/NextcloudSacmData/sacm.av/files/Recordings/2023-07-22/SQ-ST036_Ancient of Days.mp3 - 02:01
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2023-06-17/(SAT) SQ-ST020.mp3 - 03:25
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2023-05-06/(SAT) pnw1.mp3 - 03:12


In [44]:
lyrics = ""
for filepath in tqdm(split_to_limit(f"{root}/2023-03-25/[SAT] SQ-ST176.mp3")):
    print(filepath)
    audio_file = open(filepath, "rb")
    transcription = client.audio.transcriptions.create(
        # model="gpt-4o-transcribe", 
        model="whisper-1",
        file=audio_file,
        language="zh",
    )
    lyrics += transcription.text
len(lyrics), lyrics

  0%|          | 0/1 [00:00<?, ?it/s]

/mnt/NextcloudSacmData/sacm.av/files/Recordings/2023-03-25/[SAT] SQ-ST176.mp3


100%|██████████| 1/1 [00:09<00:00,  9.92s/it]


(419,
 '詞曲 李宗盛 曲 李宗盛 詞 李宗盛 曲 李宗盛 詞 李宗盛 曲 李宗盛 詞 李宗盛 曲 李宗盛 詞 李宗盛 可他我主 上帝先留 為我受死哀痛 祈願生命 恕我前用 我仍卑微無用 在我諸世之間 我先看出恩光 脆脆的從我心間脫落 在身偕我心 顏明亮心安康 如今我終日暢歡樂 乘載舟逐影我最深 北頂孤家殘溪 我的恩典 我的慈愛 無盡無窮來襲 在我諸世之間 我先看出恩光 脆重的從我心間脫落 在身偕我心 顏明亮心安康 如今我終日暢歡樂 枕身哀慈 手仍踮踏 深深在十字架 太陽暗淡 如不染寬 低沉如同鏡花 在我諸世之間 我先看出恩光 脆重的從我心間脫落 在身偕我心 顏明亮心安康 如今我終日暢歡樂 我雖淚下 如雨一般 祝我不能重返 唯有將聲 弦與竹簽 一生尊心主導 在我諸世之間 我先看出恩光 脆重的從我心間脫落 在身偕我心 顏明亮心安康 如今我終日暢歡樂 在我諸世之間 我先看出恩光 脆重的從我心間脫落 在身偕我心 顏明亮心安康 如今我終日暢歡樂')

In [45]:
titles = {}
title_to_last_chunk_idx = {}

query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

chunk_size = 120
for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
    chunk = query_lyrics[start:start + chunk_size]
    if len(chunk) < 50:
        continue
    query_py = pinyin(chunk)
    scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
    best_idx = np.argmax(scores)
    best_title = df.iloc[best_idx]["title"]
    best_score = scores[best_idx]
    print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
    if best_title not in titles:
        titles[best_title] = best_score
    else:
        boost = (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2
        titles[best_title] = max(titles[best_title], best_score) * (1.2 if boost else 1)
    title_to_last_chunk_idx[best_title] = i

print(f"{titles=}")
final_titles = [title for title, score in titles.items() if score > 0.7]
print(f"Songs: {'_'.join(final_titles)}")

[0:120] best_title='我愿深切爱主', best_score=0.5638432364096081
[120:240] best_title='可叹我主圣体血流 ', best_score=0.6707768187422934
[240:360] best_title='可叹我主圣体血流 ', best_score=0.6961869618696187
[360:480] best_title='可叹我主圣体血流 ', best_score=0.8025751072961372
titles={'我愿深切爱主': 0.5638432364096081, '可叹我主圣体血流 ': 1.002509225092251}
Songs: 可叹我主圣体血流 


In [42]:
print(df.loc[df.title.str.contains("可叹我主")].iloc[0].lyrics)

可叹我主，圣体血流，
为我受死衷痛，
情愿舍命，赎我愆尤，
我仍卑微无用。

在我主，十字架，
我先看主恩光，
罪重担，从我心皆脱落；
在圣架，我信，眼明亮，心安康，
如今，我终日常欢乐。

诚哉救主，因我罪深，
被钉苦架叹息，
莫大恩典，莫大慈心，
无尽无穷爱惜。

真神爱子受人鞭打，
身悬在十字架，
太阳暗藏，如不忍观，
地震如同惊怕。

我虽泪下如雨一般，
主恩不能酬报，
惟有将身，献於主前，
一生遵行主道。


In [ ]:
df.loc[df.title == "除你以外"]
lyrics = """至圣之主受重创，
希世痛苦难当，
遍压荆冠皆耻辱，
讥评，嫌怨，忧伤，
仰瞻慈容何惨淡？
想见满怀凄怆！
此刻愁云掩圣范，
当年基督辉光。

我主你受尽苦楚，
罪人得蒙恩典，
你受创痛至死亡，
是因我的罪愆；
思念苦刑我当受，
俯伏我主脚边；
恳求继续赐恩典，
愿常瞻仰圣颜。

我用何词作谢颂，
如斯恩谊丰隆，
成仁临难之悲哀，
无量慈怜恩宠，
恳求收我为弟子，
忠爱永不变更！
千万千万莫容我
离开主爱偷生。"""
df.at[356, "lyrics"] = lyrics
df.at[356, "pinyin"] = pinyin(lyrics)
df.to_csv("songs.csv", index=False)

In [ ]:
chunk = query_lyrics[2640:2760]
print("Query:", chunk)
query_pinyin = pinyin(chunk)
for t in ["宁静谷"]:
    inds = df.loc[df.title == t].index
    for idx in inds:
        print(f"[{idx}] {t}: {df.pinyin[idx]}")
        fuzz_score = best_window_score(query_pinyin, df.pinyin[idx], size=100, step=3)
        print(f"{fuzz_score}")

Query:  我学会了信靠他 依靠他 有一次当我 向一位朋友 倾诉我的挣扎时 他推荐我 他推荐给我一首 藏民之群的歌 叫《宁静谷》 歌词中写道 生活中的仓促 生命里的难处 只愿向他来倾诉 平安祝福在这谷 我觉得这首歌 正好讲述了 那段时期 上帝如何 把
[77] 宁静谷: zai wo xin ling shen chu you yi zuo ning jing gu wo he wo qin ai de zhu zai qi zhong an ran man bu sheng huo zhong de cang cu sheng ming li de nan chu zhi yuan xiang ta lai qing su ping an zhu fu zai zhe gu wo yu wo zhu xiang yue zhi chu chang yang zhe fen ning jing an xiang jiu xiang shi zai tian tang wo yu wo zhu xiang yue zhi chu zhu ling wo guo si yin you gu shi wo xi le zou ren sheng lu
score=0.5467158003484595, fuzz_score=0.5852417302798982, 0.2868419756502333


In [ ]:
def get_titles(filepath):
    print(f"Processing {filepath}")

    cropped_paths = split_to_limit(filepath)
    lyrics = ""
    for filepath in tqdm(cropped_paths):
        audio_file = open(filepath, "rb")
        transcription = client.audio.transcriptions.create(
            # model="gpt-4o-transcribe", 
            model="whisper-1", 
            file=audio_file,
            language="zh",
        )
        lyrics += transcription.text

    if not lyrics:
        return []

    titles = {}
    title_to_last_chunk_idx = {}

    query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

    chunk_size = 120
    for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
        chunk = query_lyrics[start:start + chunk_size]
        if len(chunk) < 50:
            continue
        query_py = pinyin(chunk)
        scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
        best_idx = np.argmax(scores)
        best_title = df.iloc[best_idx]["title"]
        best_score = scores[best_idx]
        print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
        if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
            titles[best_title] = max(titles[best_title], best_score) * 1.2
        else:
            titles[best_title] = best_score
        title_to_last_chunk_idx[best_title] = i

    print(f"{titles=}")
    duration = get_duration(filepath)
    if duration > 3 * 60:
        final_titles = [title for title, score in titles.items() if score > 0.7]
    else:
        best_title = max(titles, key=titles.get)
        final_titles = [best_title] if titles[best_title] > 0.7 else []
    return final_titles

In [ ]:
root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings/Past Events/70th ann. rec"
for f in os.listdir(root):
    final_titles = get_titles(f"{root}/{f}")
    print(f"{d}/{f}: {'_'.join(final_titles)}")

In [ ]:
for d in ["2026-06-27", "2026-06-20", "2026-06-14", "2026-05-07", "2026-04-11", "2026-03-08", "2026-03-07", "2026-02-14", "2026-02-08", "2026-02-07", "2026-02-01", "2026-01-25"]:
    matches = match_zoom_to_sq(d, verbose=False)
    print(d, json.dumps(matches, indent=2, ensure_ascii=False))